In [ ]:
import numpy as np
import mne
from scipy.io import loadmat
from scipy.signal import welch
import matplotlib.pyplot as plt
from MFDFA import MFDFA
import glob
from scipy.stats import zscore

# 1. Load and Preprocess Data
def load_bci_data(file_path):
    """Load a single .mat file and return BCI structure."""
    try:
        mat = loadmat(file_path)
        bci = mat['BCI']
        if isinstance(bci, np.ndarray) and bci.shape == (1, 1):
            bci = bci[0, 0]
        print(f"Loaded {file_path}: data shape={bci['data'].shape}, TrialData shape={bci['TrialData'].shape}")
        return bci
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None

def detect_artifacts(bci):
    """Detect bad channels (variance > 5z) and artifacted trials (>100 μV)."""
    try:
        sfreq = bci['SRATE'][0, 0]
        n_trials = bci['data'].shape[1]
        print(f"Processing {n_trials} trials with sfreq={sfreq}")
        bad_chan = []
        artifact_bool = np.zeros(n_trials, dtype=bool)
        max_amplitudes = []  # For histogram
        
        # Process each run
        runs = np.unique([bci['TrialData'][0, i]['runnumber'][0, 0] for i in range(n_trials)])
        print(f"Found {len(runs)} runs: {runs}")
        for run in runs:
            run_ind = [i for i in range(n_trials) if bci['TrialData'][0, i]['runnumber'][0, 0] == run]
            print(f"Run {run}: {len(run_ind)} trials")
            trials = bci['data'][0, run_ind]  # Shape: (n_run_trials, n_channels, n_samples)
            
            # Flatten trials for variance calculation
            trials_full = np.concatenate([t for t in trials], axis=1)  # Shape: (n_channels, total_samples)
            trials_full = trials_full.astype(np.float64)  # Cast to float64
            print(f"Run {run}: trials_full shape={trials_full.shape}")
            
            # Bandpass filter (8-30 Hz)
            trials_full_mne = mne.filter.filter_data(trials_full, sfreq, 8, 30, method='fir')
            
            # Detect bad channels
            elec_var = np.var(trials_full_mne, axis=1)
            elec_z = zscore(elec_var)
            bad_chan_run = np.where(elec_z > 5)[0]
            bad_chan.extend(bad_chan_run)
            print(f"Run {run}: bad channels={bad_chan_run}")
            
            # Detect artifacts in trials
            for i, trial_idx in enumerate(run_ind):
                trial_data = trials[i].astype(np.float64)  # Cast to float64
                trial_data_mne = mne.filter.filter_data(trial_data, sfreq, 8, 30, method='fir')
                trial_data_mne[bad_chan_run, :] = 0  # Zero out bad channels
                # Check feedback period (4000 ms to resultind)
                resultind = bci['TrialData'][0, trial_idx]['resultind'][0, 0]
                feedback_start = int(4 * sfreq)
                feedback_end = int(resultind)
                if feedback_end > trial_data_mne.shape[1]:
                    print(f"Warning: trial {trial_idx + 1} resultind={resultind} exceeds data length={trial_data_mne.shape[1]}, clipping")
                    feedback_end = trial_data_mne.shape[1]
                feedback_data = trial_data_mne[:, feedback_start:feedback_end]
                max_ampl = np.max(np.abs(feedback_data))
                max_amplitudes.append(max_ampl * 1e6)  # Convert to μV
                artifact_bool[trial_idx] = max_ampl > 100e-6  # μV to V
        
        bad_chan = np.unique(bad_chan).tolist()
        return bad_chan, artifact_bool, max_amplitudes
    except Exception as e:
        print(f"Error in detect_artifacts: {e}")
        return [], np.zeros(n_trials, dtype=bool), []

def create_custom_montage(bci):
    """Create a custom DigMontage from chaninfo electrode positions."""
    try:
        electrodes = bci['chaninfo'][0, 0]['electrodes']
        ch_names = []
        pos_list = []
        for i, e in enumerate(electrodes[0]):  # Iterate over electrodes[0]
            label = e['label'][0] if isinstance(e['label'], np.ndarray) else e['label']
            if label == 'REF.':  # Skip reference channel
                continue
            print(f"Electrode {i}: label={label}, X shape={e['X'].shape}, Y shape={e['Y'].shape}, Z shape={e['Z'].shape}")
            x = float(e['X'][0, 0])  # Extract scalar from (1,1) array
            y = float(e['Y'][0, 0])
            z = float(e['Z'][0, 0])
            ch_names.append(label)
            pos_list.append([x, y, z])
        
        pos = np.array(pos_list)  # Shape: (n_channels, 3)
        print(f"Created montage with {len(ch_names)} channels and position shape: {pos.shape}")
        if pos.shape[0] != len(ch_names):
            raise ValueError(f"Mismatch between channel names ({len(ch_names)}) and positions ({pos.shape[0]})")
        
        montage = mne.channels.make_dig_montage(
            ch_pos=dict(zip(ch_names, pos)),
            coord_frame='head'
        )
        return montage, ch_names
    except (IndexError, KeyError, ValueError) as e:
        print(f"Error creating custom montage: {e}")
        return None, None

def create_mne_raw(bci, trial_idx=0):
    """Create MNE Raw object for a single trial."""
    montage, ch_names = create_custom_montage(bci)
    if montage is None or ch_names is None:
        raise ValueError("Failed to create custom montage")
    
    # Get EEG data
    try:
        eeg_data = bci['data'][0, trial_idx]
        print(f"Trial {trial_idx + 1}: eeg_data shape={eeg_data.shape}")
        if eeg_data.ndim == 1:
            n_channels = len(ch_names)
            n_samples = eeg_data.size // n_channels
            if n_samples * n_channels == eeg_data.size:
                eeg_data = eeg_data.reshape(n_channels, n_samples)
            else:
                raise ValueError(f"Cannot reshape 1D data of size {eeg_data.size} into (n_channels={n_channels}, n_samples)")
        elif eeg_data.ndim == 3 and eeg_data.shape[0] == 1:
            eeg_data = eeg_data[0]
        if eeg_data.ndim != 2 or eeg_data.shape[0] != len(ch_names):
            raise ValueError(f"EEG data shape {eeg_data.shape} does not match n_channels={len(ch_names)}")
    except (IndexError, KeyError, ValueError) as e:
        print(f"Error accessing EEG data: {e}")
        raise
    
    sfreq = bci['SRATE'][0, 0]
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg')
    raw = mne.io.RawArray(eeg_data * 1e-6, info)  # Convert μV to V
    raw.set_montage(montage)
    
    # Filter for alpha band (8-14 Hz)
    raw.filter(8, 14, fir_design='firwin')
    
    # Mark bad channels
    try:
        noisy_channels = bci['chaninfo'][0, 0]['noisechan']
        if len(noisy_channels) > 0:
            bad_channels = [ch_names[i] for i in noisy_channels[0]]
            raw.info['bads'] = bad_channels
            raw.interpolate_bads()
    except (IndexError, KeyError):
        print("No noisy channels found or error accessing noisechan.")
    
    return raw

# 2. Visualizations
def plot_artifact_histogram(max_amplitudes, subject_id='S1'):
    """Plot histogram of maximum amplitudes (like Figure 1)."""
    plt.figure(figsize=(8, 5))
    plt.hist(max_amplitudes, bins=50, color='blue', alpha=0.7)
    plt.axvline(100, color='red', linestyle='--', label='100 μV threshold')
    plt.xlabel('Maximum Amplitude (μV)')
    plt.ylabel('Number of Trials')
    plt.title(f'Artifact Detection: Maximum Amplitudes for {subject_id}')
    plt.legend()
    plt.savefig(f'artifact_histogram_{subject_id}.png')
    plt.close()

def plot_time_series(raw, channels=['C3', 'C4']):
    """Plot EEG time series for selected channels."""
    try:
        raw.plot(scalings='auto', n_channels=2, show=False, block=False, picks=channels)
        plt.title('EEG Time Series (C3, C4)')
        plt.savefig('eeg_time_series.png')
        plt.close()
    except ValueError as e:
        print(f"Error plotting time series: {e}. Check if channels {channels} exist.")

def plot_psd(raw, channels=['C3', 'C4']):
    """Plot PSD for selected channels."""
    try:
        raw.compute_psd(fmin=8, fmax=14, picks=channels).plot(show=False)
        plt.title('Power Spectral Density (8-14 Hz)')
        plt.savefig('psd_c3_c4.png')
        plt.close()
    except ValueError as e:
        print(f"Error plotting PSD: {e}. Check if channels {channels} exist.")

def plot_erd_ers(bci, trial_idx=0, task_number=None):
    """Plot ERD/ERS for C3/C4 during a trial, optionally filtered by task."""
    try:
        montage, ch_names = create_custom_montage(bci)
        if montage is None or ch_names is None:
            print("Cannot plot ERD/ERS: Failed to create montage")
            return
        c3_idx = ch_names.index('C3')
        c4_idx = ch_names.index('C4')
        eeg_data = bci['data'][0, trial_idx]
        if eeg_data.ndim == 1:
            n_channels = len(ch_names)
            n_samples = eeg_data.size // n_channels
            if n_samples * n_channels == eeg_data.size:
                eeg_data = eeg_data.reshape(n_channels, n_samples)
            else:
                raise ValueError(f"Cannot reshape 1D data of size {eeg_data.size} into (n_channels={n_channels}, n_samples)")
        elif eeg_data.ndim == 3 and eeg_data.shape[0] == 1:
            eeg_data = eeg_data[0]
        
        # Check task number
        if task_number is not None:
            trial_task = bci['TrialData'][0, trial_idx]['tasknumber'][0, 0]
            if trial_task != task_number:
                print(f"Trial {trial_idx + 1} is task {trial_task}, not {task_number}. Skipping.")
                return
        
        eeg_c3 = eeg_data[c3_idx, :]
        eeg_c4 = eeg_data[c4_idx, :]
    except (ValueError, KeyError, IndexError) as e:
        print(f"Error in plot_erd_ers: {e}")
        return
    
    sfreq = bci['SRATE'][0, 0]
    try:
        resultind = bci['TrialData'][0, trial_idx]['resultind'][0, 0]
    except (IndexError, KeyError):
        print(f"Error accessing TrialData.resultind for trial {trial_idx}")
        return
    
    inter_trial = slice(0, int(2 * sfreq))  # -2000 to 0 ms
    feedback = slice(int(4 * sfreq), int(resultind))  # 4000 ms to resultind
    
    freqs, psd_c3_inter = welch(eeg_c3[inter_trial], fs=sfreq, nperseg=1024)
    _, psd_c3_feed = welch(eeg_c3[feedback], fs=sfreq, nperseg=1024)
    _, psd_c4_inter = welch(eeg_c4[inter_trial], fs=sfreq, nperseg=1024)
    _, psd_c4_feed = welch(eeg_c4[feedback], fs=sfreq, nperseg=1024)
    
    alpha_idx = (freqs >= 8) & (freqs <= 14)
    erd_c3 = (np.mean(psd_c3_feed[alpha_idx]) - np.mean(psd_c3_inter[alpha_idx])) / np.mean(psd_c3_inter[alpha_idx])
    erd_c4 = (np.mean(psd_c4_feed[alpha_idx]) - np.mean(psd_c4_inter[alpha_idx])) / np.mean(psd_c4_inter[alpha_idx])
    
    plt.figure(figsize=(6, 4))
    plt.bar(['C3', 'C4'], [erd_c3, erd_c4], color=['red', 'blue'])
    plt.ylabel('ERD/ERS (Relative Power Change)')
    task_name = {1: 'LR', 2: 'UD', 3: '2D'}.get(task_number, f'Trial {trial_idx + 1}')
    plt.title(f'ERD/ERS for {task_name}')
    plt.savefig(f'erd_ers_task_{task_name}.png')
    plt.close()

def plot_topography(bci, trial_idx=0, task_number=None):
    """Plot topographic map of alpha power."""
    try:
        montage, ch_names = create_custom_montage(bci)
        if montage is None or ch_names is None:
            print("Cannot plot topography: Failed to create montage")
            return
        if task_number is not None:
            trial_task = bci['TrialData'][0, trial_idx]['tasknumber'][0, 0]
            if trial_task != task_number:
                print(f"Trial {trial_idx + 1} is task {trial_task}, not {task_number}. Skipping.")
                return
        eeg_data = bci['data'][0, trial_idx]
        if eeg_data.ndim == 1:
            n_channels = len(ch_names)
            n_samples = eeg_data.size // n_channels
            if n_samples * n_channels == eeg_data.size:
                eeg_data = eeg_data.reshape(n_channels, n_samples)
            else:
                raise ValueError(f"Cannot reshape 1D data of size {eeg_data.size} into (n_channels={n_channels}, n_samples)")
        elif eeg_data.ndim == 3 and eeg_data.shape[0] == 1:
            eeg_data = eeg_data[0]
    except (IndexError, KeyError, ValueError) as e:
        print(f"Error accessing EEG data for topography: {e}")
        return
    
    sfreq = bci['SRATE'][0, 0]
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg')
    raw = mne.io.RawArray(eeg_data * 1e-6, info)
    raw.set_montage(montage)
    raw.filter(8, 14)
    
    psd, freqs = mne.time_frequency.psd_welch(raw, fmin=8, fmax=14, n_per_seg=1024)
    alpha_power = psd.mean(axis=1)
    task_name = {1: 'LR', 2: 'UD', 3: '2D'}.get(task_number, f'Trial {trial_idx + 1}')
    mne.viz.plot_topomap(alpha_power, raw.info, show=False)
    plt.title(f'Alpha Power Topography ({task_name})')
    plt.savefig(f'topography_task_{task_name}.png')
    plt.close()

def compute_pvc(bci, task_number=None, target_number=None):
    """Compute Percent Valid Correct (PVC) for a specific task and optional target."""
    try:
        n_trials = bci['TrialData'].shape[1]
        results = []
        for i in range(n_trials):
            task_match = task_number is None or bci['TrialData'][0, i]['tasknumber'][0, 0] == task_number
            target_match = target_number is None or bci['TrialData'][0, i]['targetnumber'][0, 0] == target_number
            if task_match and target_match:
                results.append(bci['TrialData'][0, i]['result'][0, 0])
        results = np.array(results)
        hits = np.sum(results == 1)
        total = len(results)
        print(f"PVC for task={task_number}, target={target_number}: {hits}/{total} trials")
        return hits / total * 100 if total > 0 else 0
    except (IndexError, KeyError) as e:
        print(f"Error computing PVC: {e}")
        return None

def compute_proficiency(bci, task_number=None, threshold=70):
    """Compute proficiency (PVC > threshold) for a task."""
    pvc = compute_pvc(bci, task_number)
    if pvc is None:
        return False
    return pvc > threshold

def plot_pvc_across_subjects(data_path, task_number=1):
    """Plot PVC across subjects for a specific task (like Figure 2)."""
    pvc_data = []
    subject_ids = [f'S{i}' for i in range(1, 21)]
    for subject_id in subject_ids:
        file_path = f'{data_path}/{subject_id}_Session_1.mat'
        bci = load_bci_data(file_path)
        if bci is None:
            continue
        pvc = compute_pvc(bci, task_number=task_number)
        if pvc is not None:
            pvc_data.append(pvc)
    
    if pvc_data:
        plt.figure(figsize=(8, 5))
        plt.bar(subject_ids, pvc_data, color='#1e88e5')
        plt.axhline(70, color='red', linestyle='--', label='Proficiency Threshold (70%)')
        plt.xlabel('Subject')
        plt.ylabel('Percent Valid Correct (%)')
        plt.title(f'PVC for Task {["LR", "UD", "2D"][task_number-1]} Across Subjects')
        plt.legend()
        plt.xticks(rotation=45)
        plt.grid(True, axis='y')
        plt.savefig(f'pvc_task_{task_number}_subjects.png')
        plt.close()

def perform_mfdfa(bci, channel='C3', trial_idx=0, task_number=None):
    """Perform MFDFA on a single channel's EEG data."""
    try:
        montage, ch_names = create_custom_montage(bci)
        if montage is None or ch_names is None:
            print("Cannot perform MFDFA: Failed to create montage")
            return None, None, None
        if task_number is not None:
            trial_task = bci['TrialData'][0, trial_idx]['tasknumber'][0, 0]
            if trial_task != task_number:
                print(f"Trial {trial_idx + 1} is task {trial_task}, not {task_number}. Skipping.")
                return None, None, None
        ch_idx = ch_names.index(channel)
        eeg_data = bci['data'][0, trial_idx]
        if eeg_data.ndim == 1:
            n_channels = len(ch_names)
            n_samples = eeg_data.size // n_channels
            if n_samples * n_channels == eeg_data.size:
                eeg_data = eeg_data.reshape(n_channels, n_samples)
            else:
                raise ValueError(f"Cannot reshape 1D data of size {eeg_data.size} into (n_channels={n_channels}, n_samples)")
        elif eeg_data.ndim == 3 and eeg_data.shape[0] == 1:
            eeg_data = eeg_data[0]
        eeg_data = eeg_data[ch_idx, :]
    except (ValueError, KeyError, IndexError) as e:
        print(f"Error in perform_mfdfa: {e}")
        return None, None, None
    
    scales = np.arange(16, 2048, 32)
    q = np.linspace(-5, 5, 101)
    lag, dfa = MFDFA(eeg_data, scales, q)
    h_q = []
    for i in range(len(q)):
        coef = np.polyfit(np.log2(lag), np.log2(dfa[i, :]), 1)
        h_q.append(coef[0])
    
    alpha = h_q + q * np.gradient(h_q, q[1] - q[0])
    f_alpha = q * alpha - (h_q * q - np.log2(dfa[:, -1]))
    
    task_name = {1: 'LR', 2: 'UD', 3: '2D'}.get(task_number, f'Trial {trial_idx + 1}')
    plt.figure(figsize=(6, 4))
    plt.plot(alpha, f_alpha, 'b-')
    plt.xlabel('Singularity Strength (α)')
    plt.ylabel('Multifractal Spectrum f(α)')
    plt.title(f'MFDFA Spectrum for {channel} ({task_name})')
    plt.grid(True)
    plt.savefig(f'mfdfa_spectrum_{task_name}.png')
    plt.close()
    
    return alpha, f_alpha, h_q

def compare_mbsr_control(data_path, task_number=1):
    """Compare PVC between MBSR and control groups for a task."""
    mbsr_pvc = []
    control_pvc = []
    for sub_num in range(1, 21):
        file_path = f'{data_path}/S{sub_num}_Session_1.mat'
        bci = load_bci_data(file_path)
        if bci is None:
            continue
        pvc = compute_pvc(bci, task_number=task_number)
        if pvc is not None:
            try:
                is_mbsr = bci['metadata'][0, 0]['MBSRsubject'][0, 0] == 1
                if is_mbsr:
                    mbsr_pvc.append(pvc)
                else:
                    control_pvc.append(pvc)
            except (KeyError, IndexError):
                print(f"Error accessing MBSRsubject for subject {sub_num}")
    
    if mbsr_pvc and control_pvc:
        print(f"Task {task_number} ({['LR', 'UD', '2D'][task_number-1]}):")
        print(f"MBSR group PVC: {np.mean(mbsr_pvc):.2f} ± {np.std(mbsr_pvc):.2f} (n={len(mbsr_pvc)})")
        print(f"Control group PVC: {np.mean(control_pvc):.2f} ± {np.std(control_pvc):.2f} (n={len(control_pvc)})")
        
        plt.figure(figsize=(6, 4))
        plt.bar(['MBSR', 'Control'], [np.mean(mbsr_pvc), np.mean(control_pvc)], 
                yerr=[np.std(mbsr_pvc), np.std(control_pvc)], color=['#1e88e5', '#d81b60'])
        plt.ylabel('Percent Valid Correct (%)')
        plt.title(f'PVC Comparison for Task {["LR", "UD", "2D"][task_number-1]}')
        plt.savefig(f'pvc_mbsr_vs_control_task_{task_number}.png')
        plt.close()
    else:
        print(f"No valid PVC data for task {task_number}")

# Main execution
if __name__ == '__main__':
    data_path = '.'  # Replace with your data directory
    subject_id = 'S1'
    try:
        # Load sample file
        file_path = f'{data_path}/{subject_id}_Session_1.mat'
        bci = load_bci_data(file_path)
        if bci is None:
            raise ValueError(f"Failed to load {file_path}")
        
        # Detect artifacts
        bad_chan, artifact_bool, max_amplitudes = detect_artifacts(bci)
        print(f"Bad channels: {bad_chan}")
        print(f"Artifacted trials: {np.where(artifact_bool)[0].tolist()}")
        plot_artifact_histogram(max_amplitudes, subject_id)
        
        # Create MNE Raw object for a valid trial
        valid_trials = [i for i in range(bci['TrialData'].shape[1]) if not artifact_bool[i] and bci['TrialData'][0, i]['result'][0, 0] == 1]
        if valid_trials:
            trial_idx = valid_trials[0]
            raw = create_mne_raw(bci, trial_idx=trial_idx)
            
            # Generate visualizations for each task
            for task in [1, 2, 3]:
                task_trials = [i for i in valid_trials if bci['TrialData'][0, i]['tasknumber'][0, 0] == task]
                if task_trials:
                    plot_erd_ers(bci, trial_idx=task_trials[0], task_number=task)
                    plot_topography(bci, trial_idx=task_trials[0], task_number=task)
                    perform_mfdfa(bci, channel='C3', trial_idx=task_trials[0], task_number=task)
        
        # Plot PVC across subjects
        for task in [1, 2, 3]:
            plot_pvc_across_subjects(data_path, task_number=task)
        
        # Compare MBSR vs. Control
        for task in [1, 2, 3]:
            compare_mbsr_control(data_path, task_number=task)
        
    except Exception as e:
        print(f"Error in main execution: {e}")

Loaded ./S1_Session_1.mat: data shape=(1, 450), TrialData shape=(1, 450)
Processing 450 trials with sfreq=1000
Found 18 runs: [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18]
Run 1: 25 trials
Run 1: trials_full shape=(62, 256344)
Setting up band-pass filter from 8 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 8.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 7.00 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 1651 samples (1.651 s)



[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s


Run 1: bad channels=[]
Setting up band-pass filter from 8 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 8.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 7.00 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 1651 samples (1.651 s)

Setting up band-pass filter from 8 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 8.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 7.00 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff fre

[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s


FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 8.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 7.00 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 1651 samples (1.651 s)

Setting up band-pass filter from 8 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 8.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 7.00 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 1651 samples (1.651 s)

Setting 

...

- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 1651 samples (1.651 s)

Setting up band-pass filter from 8 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 8.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 7.00 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 1651 samples (1.651 s)

Setting up band-pass filter from 8 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 8.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 7.00 

[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s


Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 8.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 7.00 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 1651 samples (1.651 s)

Setting up band-pass filter from 8 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 8.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 7.00 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 1651 samples (1.651 s)

Setting up band-pass filter from 8 - 30 Hz

FIR filt

[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s


Error loading ./S1_Session_1.mat: 
Error loading ./S2_Session_1.mat: 
Error loading ./S3_Session_1.mat: 
Error loading ./S4_Session_1.mat: 
Error loading ./S5_Session_1.mat: 
Error loading ./S6_Session_1.mat: 
Error loading ./S7_Session_1.mat: 
Error loading ./S8_Session_1.mat: 
Error loading ./S9_Session_1.mat: 
Error loading ./S10_Session_1.mat: 
